# Labeler Test

Tests the interactive labeling interface.

First builds a few stacks with `sattile_stack()` and saves them as `.nc` files,
then launches the labeler on those files.

In [3]:
import sys, os
sys.path.insert(0, "..")

NC_DIR = "../labeling/test_stacks"
os.makedirs(NC_DIR, exist_ok=True)

import pystac_client
import planetary_computer

from sat_tile_stack import sattile_stack, write_netcdf_from_da

print("Imports OK")

Imports OK


## Build test stacks

Build 3 small stacks from different Greenland lake locations and save as `.nc` files.

In [4]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# Three lake locations in CW Greenland
lakes = {
    "lake_A": (-49.4957, 68.7249),
    "lake_B": (-48.4330, 69.1286),
    "lake_C": (-49.1200, 68.9500),
}

TIME_RANGE = "2019-06-01/2019-06-14"

for name, centroid in lakes.items():
    outfile = f"{NC_DIR}/{name}.nc"
    if os.path.exists(outfile):
        print(f"  {name} already exists, skipping")
        continue
    
    print(f"Building {name} at {centroid}...")
    stack = sattile_stack(
        catalog, centroid,
        band_names=["B04", "B03", "B02"],
        collection="sentinel-2-l2a",
        pix_res=10, tile_size=128,
        time_range=TIME_RANGE,
        normalize=False,
        cloudmask=False,
        pull_to_mem=True,
    )
    write_netcdf_from_da(stack, outfile)

from pathlib import Path
nc_files = sorted(Path(NC_DIR).glob("*.nc"))
print(f"\nReady: {len(nc_files)} stacks in {NC_DIR}")
for f in nc_files:
    print(f"  {f.name}")

Building lake_A at (-49.4957, 68.7249)...
Pulling stack into memory, shape: (14, 3, 128, 128)
[########################################] | 100% Completed | 6.49 ss
Stack loaded, shape: (14, 3, 128, 128)
wrote ../labeling/test_stacks/lake_A.nc
Building lake_B at (-48.433, 69.1286)...
Pulling stack into memory, shape: (14, 3, 128, 128)
[########################################] | 100% Completed | 5.86 ss
Stack loaded, shape: (14, 3, 128, 128)
wrote ../labeling/test_stacks/lake_B.nc
Building lake_C at (-49.12, 68.95)...
Pulling stack into memory, shape: (14, 3, 128, 128)
[########################################] | 100% Completed | 211.85 ms
Stack loaded, shape: (14, 3, 128, 128)
wrote ../labeling/test_stacks/lake_C.nc

Ready: 3 stacks in ../labeling/test_stacks
  lake_A.nc
  lake_B.nc
  lake_C.nc


## Launch the labeler

This pops out a standalone matplotlib window with:
- **Image**: current frame with date
- **Slider**: scrub through ALL timesteps (including no-data days)
- **Radio buttons**: select class
- **Notes**: free text
- **Buttons**: Back / Skip / Submit & Next
- **Keyboard**: left/right arrows for frames, enter to submit
- **Progress bar** and **pie chart** in the corner

Close the window when done.

In [ ]:
print("Test stacks saved to labeling/test_stacks/")
print("To launch the labeler GUI, run from the repo root:")
print()
print("  python labeling/run_labeler.py --nc_dir labeling/test_stacks")

## Check saved labels

In [ ]:
import pandas as pd
from pathlib import Path

LABELS_CSV = "../labeling/labels/labels.csv"
if Path(LABELS_CSV).exists():
    df = pd.read_csv(LABELS_CSV)
    print(f"Labels saved: {len(df)} rows")
    print(df)
else:
    print("No labels saved yet.")